

**Instructor:** Luciano Argolo  
**Web:** lucianoargolo.com

---

## 🎯 Objetivos de esta clase

1. Entender la estructura de **Unity Catalog** (Catálogo → Esquema → Tabla/Volume)
2. Crear tu propio **catálogo, esquema y volume**
3. Cargar datos crudos desde un CSV
4. Realizar un **EDA (Exploratory Data Analysis)** completo
5. Aplicar **CTEs** y **Window Functions** en análisis real
6. Identificar problemas de calidad de datos

---

## 📊 Dataset: Propiedades Inmobiliarias

Trabajaremos con datos **reales y crudos** de propiedades inmobiliarias scrapeadas de:
- ZonaProp
- MercadoLibre
- Argenprop

**Este es el escenario real de un Data Engineer:** recibir datos "sucios" y analizarlos antes de procesarlos.


================================================================

## ¿Qué es Unity Catalog?

**Unity Catalog** es el sistema de gobernanza de datos de Databricks que permite:
- Organizar datos en una jerarquía clara
- Controlar accesos y permisos
- Rastrear linaje de datos
- Compartir datos entre workspaces

---

## Jerarquía de Unity Catalog

```
METASTORE (nivel más alto - administrado por Databricks)
    └── CATALOG (catálogo - como una base de datos)
            └── SCHEMA (esquema - agrupación lógica)
                    ├── TABLE (tablas de datos)
                    ├── VIEW (vistas)
                    └── VOLUME (almacenamiento de archivos)
```

### Ejemplo práctico:
```
bootcamp
    ├── landing
    │       └── archivos (volume para CSVs - simula data lake)
    └── bronze
            └── properties_bronze (tabla con datos crudos)
```

---

## 🔑 Concepto clave: VOLUME

Un **Volume** es un contenedor de archivos dentro de Unity Catalog. Es como una carpeta donde podés subir:
- CSVs
- JSONs
- Parquets
- Cualquier archivo

**Ventaja:** Los archivos quedan gobernados por Unity Catalog (permisos, auditoría, etc.)


## 📚 Tablas MANAGED vs EXTERNAL

### MANAGED TABLE (Tabla Administrada)
- Databricks **controla tanto los metadatos como los datos**
- Si borras la tabla, **se borran los datos también**
- Los datos se guardan en el storage por defecto del metastore
- **Recomendado para:** Datos procesados, tablas Silver/Gold

```sql
-- Ejemplo de tabla MANAGED
CREATE TABLE mi_catalogo.raw.mi_tabla (
    id INT,
    nombre STRING
);
-- Si haces DROP TABLE, los datos se eliminan
```

---

### EXTERNAL TABLE (Tabla Externa)
- Databricks **solo controla los metadatos**
- Los datos viven en una ubicación externa (S3, ADLS, etc.)
- Si borras la tabla, **los datos NO se borran**
- **Recomendado para:** Datos crudos, archivos compartidos, datos que no querés perder

```sql
-- Ejemplo de tabla EXTERNAL
CREATE TABLE mi_catalogo.bronze.mi_tabla_externa
USING CSV
LOCATION '/Volumes/mi_catalogo/bronze/archivos/datos.csv';
-- Si haces DROP TABLE, el archivo CSV sigue existiendo
```

---

### ¿Cuándo usar cada una?

| Escenario | Tipo recomendado |
|-----------|------------------|
| Datos crudos/bronze | EXTERNAL |
| Datos procesados (silver/gold) | MANAGED |
| Archivos compartidos entre equipos | EXTERNAL |
| Tablas temporales de análisis | MANAGED |
| Datos que vienen de otro sistema | EXTERNAL |


# ================================================================
# MÓDULO 2: CREAR TU CATÁLOGO, ESQUEMA Y VOLUME
# ================================================================

## Paso 1: Crear tu Catálogo

Cada alumno crea su propio catálogo con un nombre único.

**Convención de nombres sugerida:** `bootcamp` o `de_tunombre`


In [0]:
%sql
-- ============================================================
-- PASO 1: Crear tu catálogo
-- ============================================================
-- ⚠️ IMPORTANTE: Reemplazá 'tunombre' con tu nombre real
-- Ejemplo: bootcamp_luciano, bootcamp_maria, etc.

CREATE CATALOG IF NOT EXISTS bootcamp;

-- Verificar que se creó correctamente
SHOW CATALOGS;


catalog
bootcamp
samples
system
workspace


## Paso 2: Crear los esquemas Landing y Bronze

- El esquema `landing` almacenará los archivos crudos (CSV) — simula el data lake
- El esquema `bronze` contendrá las tablas crudas sin procesar


In [0]:
%sql
-- ============================================================
-- PASO 2: Crear los esquemas Landing y Bronze
-- ============================================================

-- Usar tu catálogo
USE CATALOG bootcamp;

-- Crear esquema Landing (archivos crudos - simula data lake)
CREATE SCHEMA IF NOT EXISTS bootcamp.landing
COMMENT 'Esquema para archivos crudos (simula data lake)';

-- Crear esquema Bronze (tablas crudas sin procesar)
CREATE SCHEMA IF NOT EXISTS bootcamp.bronze
COMMENT 'Esquema para datos crudos sin procesar (Bronze layer)';

-- Verificar
SHOW SCHEMAS;


databaseName
bronze
default
information_schema
landing
practice


## Paso 3: Crear el Volume para archivos

Un **Volume** es donde subiremos nuestro archivo CSV.


In [0]:
%sql
-- ============================================================
-- PASO 3: Crear el Volume para archivos
-- ============================================================

-- Crear volume tipo MANAGED (Databricks administra el storage)
CREATE VOLUME IF NOT EXISTS bootcamp.landing.archivos
COMMENT 'Volume para almacenar archivos CSV crudos';

-- Verificar que se creó
SHOW VOLUMES IN bootcamp.landing;


database,volume_name
landing,archivos


## Paso 4: Subir el archivo CSV al Volume

### 📤 Instrucciones para subir el archivo:

1. En el panel izquierdo de Databricks, hacé clic en **"Catalog"**
2. Navegá hasta tu catálogo → `landing` → `archivos` (el volume que creamos)
3. Hacé clic en el volume `archivos`
4. Verás un botón **"Upload to this volume"** o **"Upload"**
5. Arrastrá o seleccioná el archivo `properties_raw.csv`
6. Esperá a que termine la subida (puede tardar unos minutos por el tamaño)

### 📍 Ruta del archivo después de subir:
```
/Volumes/bootcamp/landing/archivos/properties_raw.csv
```

### ⚠️ Importante:
- El archivo pesa ~200MB, puede tardar unos minutos en subir
- Asegurate de que el nombre del archivo sea exactamente `properties_raw.csv`


In [0]:
%sql
-- ============================================================
-- PASO 4: Verificar que el archivo se subió correctamente
-- ============================================================

-- Listar archivos en el volume
LIST '/Volumes/bootcamp/landing/archivos/';


path,name,size,modification_time
/Volumes/bootcamp/landing/archivos/properties_raw.csv,properties_raw.csv,368203169,1773666913000


## Paso 5: Crear la tabla properties_bronze

Ahora vamos a crear una tabla que lea los datos del CSV.

**Tipo de tabla:** EXTERNAL (los datos quedan en el Volume, si borramos la tabla el CSV sigue existiendo)


In [0]:
%sql
-- Leer directo del archivo es posible, y es útil también para analizarlo

 SELECT * FROM read_files(
    '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
    format => 'csv',
    header => true
  )

id,ubicacion,precio,numero,calle,expensas,tipo_de_operacion,moneda,ambientes,metros_cuadrados_totales,metros_cuadrados_cubiertos,orientacion_cardinal,orientacion_inmueble,piso,cochera,antiguedad,estado,tipo_vendedor,url,zona,fecha,hora,_rescued_data
1256979.0,"Independencia 566, Belén de Escobar, Escobar",640000.0,566.0,Independencia,null,alquiler,ARS,2.0,38.0,36.0,null,null,null,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclappa-2-ambientes-en-belen-de-escobar-57872676.html?n_src=Listado&n_pills=SUM&n_pg=2&n_pos=16,escobar,2026-01-08,13:15:22,null
1256980.0,"Corrientes y Bolivar 1000, Ingeniero Maschwitz, Escobar",750000.0,1000.0,Corrientes y Bolivar,null,alquiler,ARS,3.0,70.0,55.0,null,null,1.0,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-ing.-maschwitz-escobar-57866534.html?n_src=Listado&n_pg=2&n_pos=17,escobar,2026-01-08,13:15:22,null
1256981.0,"Sta Rosa 1800, Garín, Escobar",900.0,1800.0,Sta Rosa,340000.0,alquiler,USD,3.0,95.0,75.0,null,null,null,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-tortugas-i-pilar-56969370.html?n_src=Listado&n_pills=Encargado&n_pg=2&n_pos=18,escobar,2026-01-08,13:15:22,null
1256982.0,"Country Los Caracoles, Ingeniero Maschwitz, Escobar",650.0,null,null,450000.0,alquiler,USD,2.0,120.0,60.0,null,null,null,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclcain-casa-chalet-en-alquiler-en-ingeniero-maschwitz-57863832.html?n_src=Listado&n_pills=Parrilla&n_pg=2&n_pos=19,escobar,2026-01-08,13:15:22,null
1256983.0,"Spadaccini 1100, Belén de Escobar, Escobar",900.0,1100.0,Spadaccini,176000.0,alquiler,USD,3.0,90.0,75.0,null,frente,null,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-alquiler-3-ambientes-con-cochera-belen-de-escobar-57662616.html?n_src=Listado&n_pills=Lavadero&n_pg=2&n_pos=20,escobar,2026-01-08,13:15:22,null
1256984.0,"Entre Ríos 400, Ingeniero Maschwitz, Escobar",750000.0,400.0,Entre Ríos,null,alquiler,ARS,2.0,70.0,45.0,noreste,null,null,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-casa-2-ambientes-en-alquiler-en-ingeniero-maschwitz-57860792.html?n_src=Listado&n_pg=2&n_pos=21,escobar,2026-01-08,13:15:22,null
1256985.0,"Condominio Tortugas 1-alquiler Anual Super Oportunidad!, Garín, Escobar",900.0,null,null,369000.0,alquiler,USD,3.0,95.0,75.0,null,null,null,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-condominio-tortugas-1-57860052.html?n_src=Listado&n_pills=Lavadero&n_pg=2&n_pos=22,escobar,2026-01-08,13:15:22,null
1257027.0,"bonorino 1000, parque chacabuco, capital federal",880000.0,1000.0,bonorino,null,alquiler,ARS,3.0,92.0,null,null,frente,null,1,999.0,excelente,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-3-amb-plata-baja-con-cocheray-s-exp-58047588.html?n_src=Listado&n_pills=Laundry&n_pg=5&n_pos=4,capital-federal,2026-01-08,13:15:22,null
1256986.0,"Independencia al 500, Escobar",450000.0,500.0,Independencia,71500.0,alquiler,ARS,1.0,31.0,31.0,null,null,2.0,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-monoambiente-en-venta-con-cochera-en-escobar-57854094.html?n_src=Listado&n_pg=2&n_pos=23,escobar,2026-01-08,13:15:22,null
1256987.0,"Rivadavia 631, Belén de Escobar, Escobar",430000.0,631.0,Rivadavia,null,alquiler,ARS,2.0,40.0,40.0,null,null,0.0,null,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-57853870.html?n_src=Listado&n_pills=Parrilla&n_pg=2&n_pos=24,escobar,2026-01-08,13:15:22,null


In [0]:
%sql
-- ============================================================
-- PASO 5: Crear la tabla properties_bronze
-- ============================================================

-- Primero, eliminamos la tabla si existe (para poder recrearla)
DROP TABLE IF EXISTS bootcamp.bronze.properties_bronze;

-- Crear tabla EXTERNA leyendo el CSV y nos quedamos solo con los registros que tienen url válida
CREATE TABLE bootcamp.bronze.properties_bronze
  SELECT * FROM read_files(
    '/Volumes/bootcamp/landing/archivos/properties_raw.csv',
    format => 'csv',
    header => true
  )
  where url like 'https%'
  ;



num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Verificar que la tabla se creó correctamente
SHOW TABLES IN bootcamp.bronze;


database,tableName,isTemporary
bronze,properties_bronze,false
,_sqldf,true
,bronze_eda,true
,propiedades_clean,true


In [0]:
%sql
-- Ver la estructura de la tabla
DESCRIBE bootcamp.bronze.properties_bronze;


col_name,data_type,comment
id,double,null
ubicacion,string,null
precio,string,null
numero,string,null
calle,string,null
expensas,string,null
tipo_de_operacion,string,null
moneda,string,null
ambientes,string,null
metros_cuadrados_totales,string,null



## ¿Qué es EDA para un Data Engineer?

A diferencia de un Data Analyst que busca insights de negocio, un **Data Engineer hace EDA para:**

1. **Entender la estructura** de los datos crudos
2. **Identificar problemas de calidad** (nulos, duplicados, valores inválidos)
3. **Detectar inconsistencias** que afectarán el procesamiento
4. **Planificar las transformaciones** necesarias para la capa Silver

**Motto:** "No podés limpiar lo que no entendés"

---

## 📊 Columnas del dataset

| Columna | Descripción esperada |
|---------|---------------------|
| `id` | ID único de la propiedad |
| `ubicacion` | Dirección completa |
| `numero`, `calle` | Componentes de la dirección |
| `precio`, `expensas` | Valores monetarios |
| `tipo_de_operacion` | alquiler, venta, alquiler_temporario |
| `moneda` | USD o ARS |
| `ambientes` | Cantidad de ambientes |
| `metros_cuadrados_totales/cubiertos` | Superficies |
| `orientacion_cardinal/inmueble` | Norte/Sur, Frente/Contrafrente |
| `piso`, `cochera`, `antiguedad` | Características |
| `estado`, `tipo_vendedor` | Categorías |
| `url` | URL del aviso original |
| `zona`, `fecha`, `hora` | Metadata del scraping |


## EDA Paso 1: Exploración Inicial


In [0]:
%sql
-- ============================================================
-- EDA 1.1: ¿Cuántos registros tenemos?
-- ============================================================

SELECT COUNT(*) as total_registros
FROM bootcamp.bronze.properties_bronze;


total_registros
1246131


In [0]:
%sql
-- ============================================================
-- EDA 1.2: Explorar estructura de la tabla
-- ============================================================

DESCRIBE TABLE bootcamp.bronze.properties_bronze;


id,ubicacion,precio,numero,calle,expensas,tipo_de_operacion,moneda,ambientes,metros_cuadrados_totales,metros_cuadrados_cubiertos,orientacion_cardinal,orientacion_inmueble,piso,cochera,antiguedad,estado,tipo_vendedor,url,zona,fecha,hora,_rescued_data
1256979.0,"Independencia 566, Belén de Escobar, Escobar",640000.0,566.0,Independencia,null,alquiler,ARS,2.0,38.0,36.0,null,null,null,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclappa-2-ambientes-en-belen-de-escobar-57872676.html?n_src=Listado&n_pills=SUM&n_pg=2&n_pos=16,escobar,2026-01-08,13:15:22,null
1256980.0,"Corrientes y Bolivar 1000, Ingeniero Maschwitz, Escobar",750000.0,1000.0,Corrientes y Bolivar,null,alquiler,ARS,3.0,70.0,55.0,null,null,1.0,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-ing.-maschwitz-escobar-57866534.html?n_src=Listado&n_pg=2&n_pos=17,escobar,2026-01-08,13:15:22,null
1256981.0,"Sta Rosa 1800, Garín, Escobar",900.0,1800.0,Sta Rosa,340000.0,alquiler,USD,3.0,95.0,75.0,null,null,null,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-tortugas-i-pilar-56969370.html?n_src=Listado&n_pills=Encargado&n_pg=2&n_pos=18,escobar,2026-01-08,13:15:22,null
1256982.0,"Country Los Caracoles, Ingeniero Maschwitz, Escobar",650.0,null,null,450000.0,alquiler,USD,2.0,120.0,60.0,null,null,null,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclcain-casa-chalet-en-alquiler-en-ingeniero-maschwitz-57863832.html?n_src=Listado&n_pills=Parrilla&n_pg=2&n_pos=19,escobar,2026-01-08,13:15:22,null
1256983.0,"Spadaccini 1100, Belén de Escobar, Escobar",900.0,1100.0,Spadaccini,176000.0,alquiler,USD,3.0,90.0,75.0,null,frente,null,1,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-alquiler-3-ambientes-con-cochera-belen-de-escobar-57662616.html?n_src=Listado&n_pills=Lavadero&n_pg=2&n_pos=20,escobar,2026-01-08,13:15:22,null
1256984.0,"Entre Ríos 400, Ingeniero Maschwitz, Escobar",750000.0,400.0,Entre Ríos,null,alquiler,ARS,2.0,70.0,45.0,noreste,null,null,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-casa-2-ambientes-en-alquiler-en-ingeniero-maschwitz-57860792.html?n_src=Listado&n_pg=2&n_pos=21,escobar,2026-01-08,13:15:22,null
1256985.0,"Condominio Tortugas 1-alquiler Anual Super Oportunidad!, Garín, Escobar",900.0,null,null,369000.0,alquiler,USD,3.0,95.0,75.0,null,null,null,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-condominio-tortugas-1-57860052.html?n_src=Listado&n_pills=Lavadero&n_pg=2&n_pos=22,escobar,2026-01-08,13:15:22,null
1257027.0,"bonorino 1000, parque chacabuco, capital federal",880000.0,1000.0,bonorino,null,alquiler,ARS,3.0,92.0,null,null,frente,null,1,999.0,excelente,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-3-amb-plata-baja-con-cocheray-s-exp-58047588.html?n_src=Listado&n_pills=Laundry&n_pg=5&n_pos=4,capital-federal,2026-01-08,13:15:22,null
1256986.0,"Independencia al 500, Escobar",450000.0,500.0,Independencia,71500.0,alquiler,ARS,1.0,31.0,31.0,null,null,2.0,1,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-monoambiente-en-venta-con-cochera-en-escobar-57854094.html?n_src=Listado&n_pg=2&n_pos=23,escobar,2026-01-08,13:15:22,null
1256987.0,"Rivadavia 631, Belén de Escobar, Escobar",430000.0,631.0,Rivadavia,null,alquiler,ARS,2.0,40.0,40.0,null,null,0.0,null,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-57853870.html?n_src=Listado&n_pills=Parrilla&n_pg=2&n_pos=24,escobar,2026-01-08,13:15:22,null


In [0]:
%sql
-- ============================================================
-- EDA 1.3: Ver muestra de datos (columnas principales)
-- ============================================================

SELECT 
    id,
    ubicacion,
    precio,
    expensas,
    tipo_de_operacion,
    moneda,
    ambientes,
    metros_cuadrados_totales,
    antiguedad,
    estado,
    zona
FROM bootcamp.bronze.properties_bronze
LIMIT 10;


id,ubicacion,precio,expensas,tipo_de_operacion,moneda,ambientes,metros_cuadrados_totales,metros_cuadrados_cubiertos,antiguedad,estado,zona,fecha
1256979.0,"Independencia 566, Belén de Escobar, Escobar",640000.0,null,alquiler,ARS,2.0,38.0,36.0,999.0,null,escobar,2026-01-08
1256980.0,"Corrientes y Bolivar 1000, Ingeniero Maschwitz, Escobar",750000.0,null,alquiler,ARS,3.0,70.0,55.0,999.0,null,escobar,2026-01-08
1256981.0,"Sta Rosa 1800, Garín, Escobar",900.0,340000.0,alquiler,USD,3.0,95.0,75.0,999.0,null,escobar,2026-01-08
1256982.0,"Country Los Caracoles, Ingeniero Maschwitz, Escobar",650.0,450000.0,alquiler,USD,2.0,120.0,60.0,999.0,null,escobar,2026-01-08
1256983.0,"Spadaccini 1100, Belén de Escobar, Escobar",900.0,176000.0,alquiler,USD,3.0,90.0,75.0,999.0,null,escobar,2026-01-08
1256984.0,"Entre Ríos 400, Ingeniero Maschwitz, Escobar",750000.0,null,alquiler,ARS,2.0,70.0,45.0,999.0,null,escobar,2026-01-08
1256985.0,"Condominio Tortugas 1-alquiler Anual Super Oportunidad!, Garín, Escobar",900.0,369000.0,alquiler,USD,3.0,95.0,75.0,999.0,null,escobar,2026-01-08
1257027.0,"bonorino 1000, parque chacabuco, capital federal",880000.0,null,alquiler,ARS,3.0,92.0,null,999.0,excelente,capital-federal,2026-01-08
1256986.0,"Independencia al 500, Escobar",450000.0,71500.0,alquiler,ARS,1.0,31.0,31.0,999.0,null,escobar,2026-01-08
1256987.0,"Rivadavia 631, Belén de Escobar, Escobar",430000.0,null,alquiler,ARS,2.0,40.0,40.0,999.0,null,escobar,2026-01-08


## EDA Paso 2: Análisis de Valores Nulos

Identificar campos vacíos es **crítico** para planificar la limpieza de datos.


In [0]:
%sql
-- ============================================================
-- EDA 2.1: Contar valores nulos por columna
-- ============================================================
-- Usamos CTE para calcular el total y los nulos por columna

WITH total AS (
    SELECT COUNT(*) as n 
    FROM bootcamp.bronze.properties_bronze
),
nulos AS (
    SELECT
        COUNT(*) - COUNT(id) as nulos_id,
        COUNT(*) - COUNT(ubicacion) as nulos_ubicacion,
        COUNT(*) - COUNT(precio) as nulos_precio,
        COUNT(*) - COUNT(expensas) as nulos_expensas,
        COUNT(*) - COUNT(tipo_de_operacion) as nulos_tipo_operacion,
        COUNT(*) - COUNT(moneda) as nulos_moneda,
        COUNT(*) - COUNT(ambientes) as nulos_ambientes,
        COUNT(*) - COUNT(metros_cuadrados_totales) as nulos_m2_totales,
        COUNT(*) - COUNT(metros_cuadrados_cubiertos) as nulos_m2_cubiertos,
        COUNT(*) - COUNT(orientacion_cardinal) as nulos_orientacion_cardinal,
        COUNT(*) - COUNT(orientacion_inmueble) as nulos_orientacion_inmueble,
        COUNT(*) - COUNT(piso) as nulos_piso,
        COUNT(*) - COUNT(cochera) as nulos_cochera,
        COUNT(*) - COUNT(antiguedad) as nulos_antiguedad,
        COUNT(*) - COUNT(estado) as nulos_estado,
        COUNT(*) - COUNT(zona) as nulos_zona
    FROM bootcamp.bronze.properties_bronze
)
SELECT 
    t.n as total_registros,
    n.*
FROM total t, nulos n;


total_registros,nulos_id,nulos_ubicacion,nulos_precio,nulos_expensas,nulos_tipo_operacion,nulos_moneda,nulos_ambientes,nulos_m2_totales,nulos_m2_cubiertos,nulos_orientacion_cardinal,nulos_orientacion_inmueble,nulos_piso,nulos_cochera,nulos_antiguedad,nulos_estado,nulos_zona
1246131,0,105,7240,680182,1400,5643,13566,49536,318624,1152663,1041962,958638,743929,7953,736825,0


In [0]:
%sql
-- ============================================================
-- EDA 2.2: Porcentaje de nulos en columnas clave
-- ============================================================

WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp.bronze.properties_bronze
)
SELECT 
    ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as pct_nulos_precio,
    ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as pct_nulos_expensas,
    ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as pct_nulos_ambientes,
    ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as pct_nulos_m2_totales,
    ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as pct_nulos_m2_cubiertos,
    ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as pct_nulos_orientacion,
    ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as pct_nulos_antiguedad
FROM bootcamp.bronze.properties_bronze
CROSS JOIN totales t
GROUP BY t.total;

pct_nulos_precio,pct_nulos_expensas,pct_nulos_ambientes,pct_nulos_m2_totales,pct_nulos_m2_cubiertos,pct_nulos_orientacion,pct_nulos_antiguedad
0.58,54.58,1.09,3.98,25.57,92.50,0.64


In [0]:
%sql
-- ============================================================
-- EDA 2.3: Identificar columnas con >50% de valores nulos
-- ============================================================
-- Columnas con >50% nulos son críticas: hay que decidir si eliminarlas o imputarlas

WITH totales AS (
    SELECT COUNT(*) as total FROM bootcamp.bronze.properties_bronze
),
pct_nulos AS (
    SELECT 
        ROUND((t.total - COUNT(precio)) * 100.0 / t.total, 2) as precio,
        ROUND((t.total - COUNT(expensas)) * 100.0 / t.total, 2) as expensas,
        ROUND((t.total - COUNT(ambientes)) * 100.0 / t.total, 2) as ambientes,
        ROUND((t.total - COUNT(metros_cuadrados_totales)) * 100.0 / t.total, 2) as metros_cuadrados_totales,
        ROUND((t.total - COUNT(metros_cuadrados_cubiertos)) * 100.0 / t.total, 2) as metros_cuadrados_cubiertos,
        ROUND((t.total - COUNT(orientacion_cardinal)) * 100.0 / t.total, 2) as orientacion_cardinal,
        ROUND((t.total - COUNT(antiguedad)) * 100.0 / t.total, 2) as antiguedad,
        ROUND((t.total - COUNT(piso)) * 100.0 / t.total, 2) as piso,
        ROUND((t.total - COUNT(cochera)) * 100.0 / t.total, 2) as cochera
    FROM bootcamp.bronze.properties_bronze
    CROSS JOIN totales t
    GROUP BY t.total
)
SELECT columna, porcentaje_nulos
FROM pct_nulos
UNPIVOT (
    porcentaje_nulos FOR columna IN (
        precio, expensas, ambientes, 
        metros_cuadrados_totales, metros_cuadrados_cubiertos,
        orientacion_cardinal, antiguedad, piso, cochera
    )
)
WHERE porcentaje_nulos > 50
ORDER BY porcentaje_nulos DESC

columna,porcentaje_nulos
orientacion_cardinal,92.50
piso,76.93
cochera,59.70
expensas,54.58


## EDA Paso 3: Cardinalidad y Distribución de Categóricos

Entender qué valores tienen las columnas categóricas y su frecuencia.


In [0]:
%sql
-- ============================================================
-- EDA 3.1: Distribución de tipo_de_operacion
-- ============================================================

SELECT 
    tipo_de_operacion,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM bootcamp.bronze.properties_bronze
GROUP BY tipo_de_operacion
ORDER BY cantidad DESC;


tipo_de_operacion,cantidad,porcentaje
alquiler,626540,50.28
venta,603346,48.42
alquiler_temporal,14524,1.17
null,1400,0.11
alquiler_de_habitacion,206,0.02
alquiler_anual,35,0.00
venta/alquiler,19,0.00
alquiler_temporario,13,0.00
venta_y_alquiler,9,0.00
consultar,6,0.00


In [0]:
%sql
-- ============================================================
-- EDA 3.2: Distribución de moneda
-- ============================================================

SELECT 
    moneda,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM bootcamp.bronze.properties_bronze
GROUP BY moneda
ORDER BY cantidad DESC;


moneda,cantidad,porcentaje
USD,851579,68.34
ARS,388893,31.21
null,5643,0.45
MXN,8,0.00
consultar,2,0.00
guaranies,2,0.00
uyu,2,0.00
null,1,0.00
ar,1,0.00


In [0]:
%sql
-- ============================================================
-- EDA 3.3: Distribución de ambientes
-- ============================================================

SELECT 
    ambientes,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM bootcamp.bronze.properties_bronze
GROUP BY ambientes
ORDER BY ambientes;


ambientes,cantidad,porcentaje
null,13566,1.09
0.0,2255,0.18
1.0,156331,12.55
1.5,6,0.00
10.0,2655,0.21
11.0,641,0.05
12.0,419,0.03
123.0,1,0.00
13.0,91,0.01
14.0,182,0.01


In [0]:
%sql
-- ============================================================
-- EDA 3.4: Top 15 zonas con más propiedades
-- ============================================================

SELECT 
    zona,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM bootcamp.bronze.properties_bronze
GROUP BY zona
ORDER BY cantidad DESC
LIMIT 15;


zona,cantidad,porcentaje
capital-federal,450364,36.14
tigre,91470,7.34
pilar,80214,6.44
vicente-lopez,67555,5.42
escobar,57302,4.60
san-isidro,47597,3.82
general-san-martin,42987,3.45
san-miguel,30786,2.47
san-fernando,30643,2.46
jose-c-paz,16576,1.33


In [0]:
%sql
-- ============================================================
-- EDA 3.5: Distribución de estado de las propiedades
-- ============================================================

SELECT 
    estado,
    COUNT(*) as cantidad,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM bootcamp.bronze.properties_bronze
GROUP BY estado
ORDER BY cantidad DESC;


estado,cantidad,porcentaje
null,736825,59.13
excelente,370092,29.70
bueno,109399,8.78
a_refaccionar,26521,2.13
muy bueno,1076,0.09
a_estrenar,849,0.07
a estrenar,362,0.03
impecable,297,0.02
0,191,0.02
terminada,124,0.01


## EDA Paso 4: Estadísticas Descriptivas de Variables Numéricas

Analizar rangos, promedios y detectar outliers en las columnas numéricas.


# ¿Qué ocurre cuando los datos no están bien formateados?

> **Nota:** La siguiente celda va a fallar intencionalmente. Los datos en Bronze tienen columnas tipo `string` donde deberían ser números. Primero vemos el error, luego lo solucionamos creando una vista limpia.

In [0]:
%sql

-- ============================================================
-- EDA 4.1 (ANTES de limpiar): Estadísticas de PRECIO (datos crudos)
-- ============================================================

SELECT 
    moneda,
    COUNT(*) as cantidad,
    ROUND(MIN(precio), 2) as precio_min,
    ROUND(MAX(precio), 2) as precio_max,
    ROUND(AVG(precio), 2) as precio_promedio,
    ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,
    ROUND(PERCENTILE(precio, 0.25), 2) as precio_p25,
    ROUND(PERCENTILE(precio, 0.75), 2) as precio_p75
FROM bootcamp.bronze.properties_bronze
WHERE precio IS NOT NULL AND precio > 0
GROUP BY moneda
ORDER BY cantidad DESC;


---------------------------------------------------------------------------
NumberFormatException                     Traceback (most recent call last)
File <command-4602398801995385>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', '\n-- ============================================================\n-- EDA 4.1: Estadísticas de PRECIO (separado por moneda)\n-- ============================================================\n\nSELECT \n    moneda,\n    COUNT(*) as cantidad,\n    ROUND(MIN(precio), 2) as precio_min,\n    ROUND(MAX(precio), 2) as precio_max,\n    ROUND(AVG(precio), 2) as precio_promedio,\n    ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,\n    ROUND(PERCENTILE(precio, 0.25), 2) as precio_p25,\n    ROUND(PERCENTILE(precio, 0.75), 2) as precio_p75\nFROM bootcamp.bronze.properties_bronze\nWHERE precio IS NOT NULL AND precio > 0\nGROUP BY moneda\nORDER BY cantidad DESC;\n')

File /databricks/python/lib/python3.11/site-packages/IPython/core/interactiveshell.py:2541, i

In [0]:
%sql
-- ============================================================
-- EDA 4.2 (ANTES de limpiar): Estadísticas de METROS CUADRADOS (datos crudos)
-- ============================================================

SELECT 
    'metros_cuadrados_totales' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_totales), 2) as minimo,
    ROUND(MAX(metros_cuadrados_totales), 2) as maximo,
    ROUND(AVG(metros_cuadrados_totales), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_totales, 0.5), 2) as mediana
FROM bootcamp.bronze.properties_bronze
WHERE metros_cuadrados_totales IS NOT NULL AND metros_cuadrados_totales > 0

UNION ALL

SELECT 
    'metros_cuadrados_cubiertos' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_cubiertos), 2) as minimo,
    ROUND(MAX(metros_cuadrados_cubiertos), 2) as maximo,
    ROUND(AVG(metros_cuadrados_cubiertos), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.5), 2) as mediana
FROM bootcamp.bronze.properties_bronze
WHERE metros_cuadrados_cubiertos IS NOT NULL AND metros_cuadrados_cubiertos > 0;


## Es acá donde el Data engineer entra en juego, sin la automatización y el tratamiento adecuado de los datos, el análisis no es viable, o simplemente carece de sentido el resultado

# Corrigamos los datos por ahora de manera temporal y veamos que ocurre

In [0]:
%sql
describe bootcamp.bronze.properties_bronze

col_name,data_type,comment
id,double,null
ubicacion,string,null
precio,string,null
numero,string,null
calle,string,null
expensas,string,null
tipo_de_operacion,string,null
moneda,string,null
ambientes,string,null
metros_cuadrados_totales,string,null


In [0]:
%sql
create or replace temporary View propiedades_clean as
select 
CASE
  WHEN precio RLIKE '^[^a-zA-Z]+$' THEN precio::double
  ELSE NULL
  END as precio,
  moneda,
CASE
  WHEN ambientes RLIKE '^[^a-zA-Z]+$' THEN ambientes::double
  ELSE NULL
END as ambientes
 ,
 CASE
  WHEN metros_cuadrados_totales RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_totales::double
  ELSE NULL
END as metros_cuadrados_totales
 ,
 CASE
  WHEN metros_cuadrados_cubiertos RLIKE '^[^a-zA-Z]+$' THEN metros_cuadrados_cubiertos::double
  ELSE NULL
END as metros_cuadrados_cubiertos
 ,
 CASE
  WHEN antiguedad RLIKE '^[^a-zA-Z]+$' THEN antiguedad::double
  ELSE NULL
END as antiguedad,
tipo_de_operacion
,id
,ubicacion
,numero
,calle
,expensas
,orientacion_cardinal
,orientacion_inmueble
,piso
,cochera
,estado
,tipo_vendedor
,url
,zona
,fecha
,hora
from bootcamp.bronze.properties_bronze

## Repetimos el mismo análisis de antes para ver que pasa

In [0]:
%sql

-- ============================================================
-- EDA 4.1: Estadísticas de PRECIO (separado por moneda y tipo de operación)
-- ============================================================

SELECT 
    moneda,
    tipo_de_operacion,
    COUNT(*) as cantidad,
    ROUND(MIN(precio), 2) as precio_min,
    ROUND(MAX(precio), 2) as precio_max,
    ROUND(AVG(precio), 2) as precio_promedio,
    ROUND(PERCENTILE(precio, 0.5), 2) as precio_mediana,
    ROUND(PERCENTILE(precio, 0.25), 2) as precio_p25,
    ROUND(PERCENTILE(precio, 0.75), 2) as precio_p75
FROM propiedades_clean
WHERE precio IS NOT NULL AND precio > 0
GROUP BY moneda, tipo_de_operacion
ORDER BY cantidad DESC;


moneda,tipo_de_operacion,cantidad,precio_min,precio_max,precio_promedio,precio_mediana,precio_p25,precio_p75
USD,venta,589016,0.06,1.111111111E9,219097.52,135000.0,79500.0,241200.0
ARS,alquiler,374379,1.0,1.2E9,793701.4,650000.0,500000.0,900000.0
USD,alquiler,248213,1.0,2.14423E7,18585.05,1600.0,1000.0,3000.0
USD,alquiler_temporal,12342,1.0,4200000.0,8147.83,1300.0,700.0,3000.0
ARS,venta,9566,1.0,1.434768228E9,9169126.52,700000.0,240000.0,1400000.0
ARS,alquiler_temporal,1999,65.0,4000000.0,697957.78,690000.0,500000.0,790000.0
USD,null,857,35.0,2000000.0,139784.69,78000.0,3700.0,160000.0
null,venta,812,1.0,5000000.0,257261.63,121600.0,75000.0,216004.5
null,alquiler,775,1.0,1.1111111E7,474931.97,75000.0,1350.0,700000.0
ARS,null,486,100.0,5000000.0,872479.21,730000.0,500000.0,1000000.0


## Hallazgo: monedas inválidas, tipos de operación sucios y outliers extremos

Del análisis anterior detectamos:
- **Monedas basura:** MXN, guaranies, uyu, consultar, ar — son errores del scraping (< 15 registros en total)
- **Tipos de operación sucios:** variantes como "alquieler", "alquier", "alguilar", "venta/alquiler", etc.
- **Nulls:** registros sin moneda y sin tipo de operación
- **Outliers extremos:** precios máximos de 1,111,111,111 USD y 1,434,768,228 ARS son claramente errores

**Acción:** Filtramos solo USD y ARS, solo los 3 tipos de operación principales (venta, alquiler, alquiler_temporal), y usamos percentiles P1/P99 por moneda+operación para cortar outliers extremos.

In [0]:
%sql
-- ============================================================
-- EDA 4.1b (BONUS — no está en el PDF): Estadísticas de PRECIO (filtrado)
-- Solo USD/ARS, operaciones principales, sin outliers (P1-P99)
-- ============================================================
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT 
    p.moneda,
    p.tipo_de_operacion,
    COUNT(*) AS cantidad,
    ROUND(MIN(p.precio), 2) AS precio_min,
    ROUND(MAX(p.precio), 2) AS precio_max,
    ROUND(AVG(p.precio), 2) AS precio_promedio,
    ROUND(PERCENTILE(p.precio, 0.5), 2) AS precio_mediana,
    ROUND(PERCENTILE(p.precio, 0.25), 2) AS precio_p25,
    ROUND(PERCENTILE(p.precio, 0.75), 2) AS precio_p75
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
  GROUP BY p.moneda, p.tipo_de_operacion
  ORDER BY cantidad DESC

moneda,tipo_de_operacion,cantidad,precio_min,precio_max,precio_promedio,precio_mediana,precio_p25,precio_p75
USD,venta,577278,2300.0,1515000.0,196176.52,135000.0,80000.0,240000.0
ARS,alquiler,367201,160000.0,2800000.0,760959.57,650000.0,500000.0,900000.0
USD,alquiler,244038,400.0,550000.0,5235.92,1600.0,1000.0,2950.0
ARS,venta,9394,1200.0,1.2E8,1513914.95,700000.0,250000.0,1400000.0


In [0]:
%sql
-- ============================================================
-- Creamos la nueva vista temporal
-- ============================================================
CREATE OR REPLACE TEMPORARY VIEW propiedades_clean_2 AS
(
    WITH limites AS (
    SELECT 
      moneda,
      tipo_de_operacion,
      PERCENTILE(precio, 0.01) AS p01,
      PERCENTILE(precio, 0.99) AS p99
    FROM propiedades_clean
    WHERE precio > 0
      AND moneda IN ('USD', 'ARS')
      AND tipo_de_operacion IN ('venta', 'alquiler')
    GROUP BY moneda, tipo_de_operacion
  )
  SELECT p.*
  FROM propiedades_clean p
  JOIN limites l 
    ON p.moneda = l.moneda 
    AND p.tipo_de_operacion = l.tipo_de_operacion
  WHERE p.precio BETWEEN l.p01 AND l.p99
);

In [0]:
%sql
-- ============================================================
-- EDA 4.2: Estadísticas de METROS CUADRADOS
-- ============================================================

SELECT 
    'metros_cuadrados_totales' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_totales), 2) as minimo,
    ROUND(MAX(metros_cuadrados_totales), 2) as maximo,
    ROUND(AVG(metros_cuadrados_totales), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_totales, 0.5), 2) as mediana
FROM propiedades_clean_2
WHERE metros_cuadrados_totales IS NOT NULL AND metros_cuadrados_totales > 0

UNION ALL

SELECT 
    'metros_cuadrados_cubiertos' as columna,
    COUNT(*) as cantidad_no_nulos,
    ROUND(MIN(metros_cuadrados_cubiertos), 2) as minimo,
    ROUND(MAX(metros_cuadrados_cubiertos), 2) as maximo,
    ROUND(AVG(metros_cuadrados_cubiertos), 2) as promedio,
    ROUND(PERCENTILE(metros_cuadrados_cubiertos, 0.5), 2) as mediana
FROM propiedades_clean_2
WHERE metros_cuadrados_cubiertos IS NOT NULL AND metros_cuadrados_cubiertos > 0;



columna,cantidad_no_nulos,minimo,maximo,promedio,mediana
metros_cuadrados_totales,1149772,1.0,2.147483647E9,9107.14,78.0
metros_cuadrados_cubiertos,889644,1.0,2.0E9,6879.64,74.0


## Ahora la query funciona, pero los datos siguen siendo de poca confianza como vimos antes, acá es donde desde Bronze a Silver empezamos a hacer limpieza 

In [0]:
%sql
-- ============================================================
-- EDA 4.3: Análisis de ANTIGÜEDAD
-- ============================================================
-- NOTA: Vimos en los datos que 999 parece ser un valor por defecto (missing)

SELECT 
    antiguedad,
    COUNT(*) as cantidad
FROM propiedades_clean_2
GROUP BY antiguedad
ORDER BY cantidad DESC
LIMIT 20;


antiguedad,cantidad
999.0,1190190
null,7668
0.0,16
1.0,12
41.0,3
9.0,2
69.0,2
46.0,2
50.0,1
61.0,1


In [0]:
%sql
-- ============================================================
-- EDA 4.4: Estadísticas de antigüedad SIN el valor 999
-- ============================================================

SELECT 
    COUNT(*) as cantidad,
    MIN(CAST(antiguedad AS INT)) as antiguedad_min,
    MAX(CAST(antiguedad AS INT)) as antiguedad_max,
    ROUND(AVG(CAST(antiguedad AS INT)), 2) as antiguedad_promedio,
    ROUND(PERCENTILE(CAST(antiguedad AS INT), 0.5), 0) as antiguedad_mediana
FROM propiedades_clean_2
WHERE antiguedad IS NOT NULL 
  AND antiguedad != '999' 
  AND antiguedad >= 0;


cantidad,antiguedad_min,antiguedad_max,antiguedad_promedio,antiguedad_mediana
52,0,81,17.79,1.0


## EDA Paso 5: Detección de Problemas de Calidad

Identificar datos que necesitarán limpieza en la capa Silver.


In [0]:
%sql
-- ============================================================
-- EDA 5.1: Reporte de calidad de datos usando CTEs
-- ============================================================

WITH total AS (
    SELECT COUNT(*) as n 
    FROM propiedades_clean_2
),
problemas AS (
    SELECT
        -- Precios problemáticos
        COUNT(CASE WHEN precio IS NULL OR precio::float <= 0 THEN 1 END) as precio_invalido,
        -- Metros cuadrados problemáticos  
        COUNT(CASE WHEN metros_cuadrados_totales IS NULL OR metros_cuadrados_totales <= 0 THEN 1 END) as m2_invalido,
        -- Antigüedad con valor 999 (placeholder)
        COUNT(CASE WHEN antiguedad = '999' OR antiguedad = 999 THEN 1 END) as antiguedad_999,
        -- Ambientes = 0 o NULL
        COUNT(CASE WHEN ambientes IS NULL OR ambientes = 0 THEN 1 END) as ambientes_invalido,
        -- Moneda vacía
        COUNT(CASE WHEN moneda IS NULL OR moneda = '' THEN 1 END) as moneda_vacia,
        -- Tipo de operación vacía
        COUNT(CASE WHEN tipo_de_operacion IS NULL OR tipo_de_operacion = '' THEN 1 END) as tipo_operacion_vacia
    FROM propiedades_clean_2
)
SELECT 
    t.n as total_registros,
    p.precio_invalido,
    ROUND(p.precio_invalido * 100.0 / t.n, 2) as pct_precio_invalido,
    p.m2_invalido,
    ROUND(p.m2_invalido * 100.0 / t.n, 2) as pct_m2_invalido,
    p.antiguedad_999,
    ROUND(p.antiguedad_999 * 100.0 / t.n, 2) as pct_antiguedad_999,
    p.ambientes_invalido,
    ROUND(p.ambientes_invalido * 100.0 / t.n, 2) as pct_ambientes_invalido
FROM total t, problemas p;


total_registros,precio_invalido,pct_precio_invalido,m2_invalido,pct_m2_invalido,antiguedad_999,pct_antiguedad_999,ambientes_invalido,pct_ambientes_invalido
1197911,0,0.00,48139,4.02,1190190,99.36,14902,1.24


In [0]:
%sql
-- ============================================================
-- EDA 5.2: Detectar posibles duplicados
-- ============================================================

WITH duplicados AS (
    SELECT 
        precio,
        url,
        COUNT(*) as veces
    FROM bootcamp.bronze.properties_bronze
    GROUP BY precio, url
    HAVING COUNT(*) > 1
)
SELECT 
    COUNT(*) as grupos_duplicados,
    SUM(veces) as total_registros_duplicados,
    SUM(veces - 1) as registros_extra_por_duplicacion
FROM duplicados;


grupos_duplicados,total_registros_duplicados,registros_extra_por_duplicacion
83188,217415,134227


In [0]:
%sql
-- ============================================================
-- EDA 5.3: Ver ejemplos de duplicados
-- ============================================================

WITH duplicados AS (
    SELECT 
        precio,
        url,
        COUNT(*) as veces
    FROM bootcamp.bronze.properties_bronze
    GROUP BY precio, url
    HAVING COUNT(*) > 1
)
SELECT *
FROM duplicados
ORDER BY veces DESC
LIMIT 10;


precio,url,veces
62000.0,https://departamento.mercadolibre.com.ar/MLA-2059467746-venta-departamento-2-ambientes-con-cochera-a-estrenar-en-villa-luzuriaga-uf-4-_JM,12
56000.0,https://departamento.mercadolibre.com.ar/MLA-1507670893-venta-departamento-3-ambientes-lateral-en-ramos-mejia-_JM,11
155000.0,https://www.argenprop.com/casa-en-venta-en-hurlingham-3-ambientes--16120684,10
55000.0,https://www.argenprop.com/departamento-en-venta-en-moreno-2-ambientes--17694403,10
95000.0,https://liderprop.com/es-ar/propiedades/5864972/venta--casa--hurlingham-hurlingham/,10
350000.0,https://www.argenprop.com/departamento-en-alquiler-en-wilde-1-ambiente--16853930,10
238000.0,https://liderprop.com/es-ar/propiedades/5870643/venta--casa--san-isidro-san-isidro/,10
21000.0,https://www.argenprop.com/casa-en-venta-en-domselaar-2-ambientes--17286651,10
300000.0,https://www.argenprop.com/departamento-en-alquiler-en-ramos-mejia-1-ambiente--17626376,10
900000.0,https://liderprop.com/es-ar/propiedades/5509769/alquiler--casa--muniz/,10


In [0]:
%sql
-- ============================================================
-- EDA 5.4: Detectar outliers extremos en precio
-- ============================================================

-- Propiedades con precios sospechosamente altos o bajos
WITH stats AS (
    SELECT 
        moneda,
        PERCENTILE(precio, 0.01) as p01,
        PERCENTILE(precio, 0.99) as p99
    FROM propiedades_clean_2
    WHERE precio > 0
    GROUP BY moneda
)
SELECT 
    p.moneda,
    'Muy bajo (< P01)' as tipo_outlier,
    COUNT(*) as cantidad
FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE p.precio < s.p01 AND p.precio > 0
GROUP BY p.moneda

UNION ALL

SELECT 
    p.moneda,
    'Muy alto (> P99)' as tipo_outlier,
    COUNT(*) as cantidad
FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE p.precio > s.p99
GROUP BY p.moneda
ORDER BY moneda, tipo_outlier;


moneda,tipo_outlier,cantidad
ARS,Muy alto (> P99),2450
ARS,Muy bajo (< P01),3484
USD,Muy alto (> P99),8213
USD,Muy bajo (< P01),4310


In [0]:
%sql
create or replace temp view bronze_EDA as (
SELECT 
    edl.id,
    edl.ubicacion,
    CASE 
        WHEN edl.precio = 'NaN' OR edl.precio NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.precio::float BETWEEN -2147483648 AND 2147483648 THEN edl.precio::float
        ELSE NULL
    END AS precio,
    CASE 
        WHEN edl.numero = 'NaN' OR edl.numero NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.numero::float BETWEEN -10000 AND 50000 THEN edl.numero::float
        ELSE NULL
    END AS numero,
    edl.calle,
    CASE
        WHEN edl.expensas = 'NaN' OR edl.expensas NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        WHEN edl.expensas::float BETWEEN -20000000 AND 20000000 THEN edl.expensas::float
        ELSE NULL
    END AS expensas,
    edl.tipo_de_operacion,
    CASE
        WHEN lower(edl.moneda) LIKE '%dolares%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%us%' THEN 'USD'
        WHEN lower(edl.moneda) LIKE '%mxn%' THEN 'MXN'
        WHEN lower(edl.moneda) LIKE '%pesos%' THEN 'ARS'
        WHEN lower(edl.moneda) LIKE '%ars%' THEN 'ARS'
        ELSE edl.moneda
    END AS moneda,
    CASE
        WHEN edl.ambientes = 'NaN' OR edl.ambientes NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.ambientes::float
    END AS ambientes,
    CASE 
        WHEN edl.metros_cuadrados_totales = 'NaN' OR edl.metros_cuadrados_totales NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_totales::decimal
    END AS metros_cuadrados_totales,
    CASE 
        WHEN edl.metros_cuadrados_cubiertos = 'NaN' OR edl.metros_cuadrados_cubiertos NOT RLIKE '^-?[0-9]+\.?[0-9]*$' THEN NULL
        ELSE edl.metros_cuadrados_cubiertos::decimal
    END AS metros_cuadrados_cubiertos,
    edl.orientacion_cardinal,
    edl.orientacion_inmueble,
    CASE
        WHEN edl.piso IS NULL THEN NULL
        WHEN edl.piso = 'NaN' THEN NULL
        ELSE edl.piso::float
    END AS piso,
    CASE 
        WHEN edl.cochera = 'tiene' THEN 1
        ELSE NULL
    END AS cochera,
    edl.antiguedad,
    edl.estado,
    edl.tipo_vendedor,
    edl.url,
    edl.zona,
    edl.fecha,
    edl.hora
FROM propiedades_clean_2 edl
);


Executing subquery: -- ============================================================
-- EDA 5.4: Detectar outliers extremos en precio
-- ============================================================

-- Propiedades con precios sospechosamente altos o bajos
WITH stats AS (
    SELECT 
        moneda,
        PERCENTILE(precio, 0.01) as p01,
        PERCENTILE(precio, 0.99) as p99
    FROM propiedades_clean_2
    WHERE precio > 0
    GROUP BY moneda
)
SELECT 
    p.moneda,
    'Muy bajo (< P01)' as tipo_outlier,
    COUNT(*) as cantidad
FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE p.precio < s.p01 AND p.precio > 0
GROUP BY p.moneda

UNION ALL

SELECT 
    p.moneda,
    'Muy alto (> P99)' as tipo_outlier,
    COUNT(*) as cantidad
FROM propiedades_clean_2 p
JOIN stats s ON p.moneda = s.moneda
WHERE p.precio > s.p99
GROUP BY p.moneda
ORDER BY moneda, tipo_outlier.
Executing subquery: create or replace temp view bronze_EDA as (
	SELECT 
			case 
				when edl.precio = '

In [0]:
%sql
select * from bronze_EDA limit 100

id,ubicacion,precio,numero,calle,expensas,tipo_de_operacion,moneda,ambientes,metros_cuadrados_totales,metros_cuadrados_cubiertos,orientacion_cardinal,orientacion_inmueble,piso,cochera,antiguedad,estado,tipo_vendedor,url,zona,fecha,hora
1256979.0,"Independencia 566, Belén de Escobar, Escobar",640000.0,566.0,Independencia,null,alquiler,ARS,2.0,38,36,null,null,null,null,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclappa-2-ambientes-en-belen-de-escobar-57872676.html?n_src=Listado&n_pills=SUM&n_pg=2&n_pos=16,escobar,2026-01-08,13:15:22
1256980.0,"Corrientes y Bolivar 1000, Ingeniero Maschwitz, Escobar",750000.0,1000.0,Corrientes y Bolivar,null,alquiler,ARS,3.0,70,55,null,null,1.0,null,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-ing.-maschwitz-escobar-57866534.html?n_src=Listado&n_pg=2&n_pos=17,escobar,2026-01-08,13:15:22
1256981.0,"Sta Rosa 1800, Garín, Escobar",900.0,1800.0,Sta Rosa,340000.0,alquiler,USD,3.0,95,75,null,null,null,null,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-tortugas-i-pilar-56969370.html?n_src=Listado&n_pills=Encargado&n_pg=2&n_pos=18,escobar,2026-01-08,13:15:22
1256982.0,"Country Los Caracoles, Ingeniero Maschwitz, Escobar",650.0,null,null,450000.0,alquiler,USD,2.0,120,60,null,null,null,null,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclcain-casa-chalet-en-alquiler-en-ingeniero-maschwitz-57863832.html?n_src=Listado&n_pills=Parrilla&n_pg=2&n_pos=19,escobar,2026-01-08,13:15:22
1256983.0,"Spadaccini 1100, Belén de Escobar, Escobar",900.0,1100.0,Spadaccini,176000.0,alquiler,USD,3.0,90,75,null,frente,null,null,999.0,null,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-alquiler-3-ambientes-con-cochera-belen-de-escobar-57662616.html?n_src=Listado&n_pills=Lavadero&n_pg=2&n_pos=20,escobar,2026-01-08,13:15:22
1256984.0,"Entre Ríos 400, Ingeniero Maschwitz, Escobar",750000.0,400.0,Entre Ríos,null,alquiler,ARS,2.0,70,45,noreste,null,null,null,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-casa-2-ambientes-en-alquiler-en-ingeniero-maschwitz-57860792.html?n_src=Listado&n_pg=2&n_pos=21,escobar,2026-01-08,13:15:22
1256985.0,"Condominio Tortugas 1-alquiler Anual Super Oportunidad!, Garín, Escobar",900.0,null,null,369000.0,alquiler,USD,3.0,95,75,null,null,null,null,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-en-condominio-tortugas-1-57860052.html?n_src=Listado&n_pills=Lavadero&n_pg=2&n_pos=22,escobar,2026-01-08,13:15:22
1257027.0,"bonorino 1000, parque chacabuco, capital federal",880000.0,1000.0,bonorino,null,alquiler,ARS,3.0,92,null,null,frente,null,null,999.0,excelente,inmobiliaria,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-3-amb-plata-baja-con-cocheray-s-exp-58047588.html?n_src=Listado&n_pills=Laundry&n_pg=5&n_pos=4,capital-federal,2026-01-08,13:15:22
1256986.0,"Independencia al 500, Escobar",450000.0,500.0,Independencia,71500.0,alquiler,ARS,1.0,31,31,null,null,2.0,null,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-monoambiente-en-venta-con-cochera-en-escobar-57854094.html?n_src=Listado&n_pg=2&n_pos=23,escobar,2026-01-08,13:15:22
1256987.0,"Rivadavia 631, Belén de Escobar, Escobar",430000.0,631.0,Rivadavia,null,alquiler,ARS,2.0,40,40,null,null,0.0,null,999.0,null,null,https://www.zonaprop.com.ar/propiedades/clasificado/alclapin-departamento-en-alquiler-57853870.html?n_src=Listado&n_pills=Parrilla&n_pg=2&n_pos=24,escobar,2026-01-08,13:15:22


## EDA Paso 6: Análisis Avanzado con CTEs y Window Functions

Combinamos todo lo aprendido para generar insights más profundos.

> Corresponde a **Parte 6** del PDF de ejercicios.


In [0]:
%sql
-- ============================================================
-- EDA 6.1: Precio promedio por zona con RANKING (ROW_NUMBER)
-- Usar CTEs para calcular y rankear zonas por precio
-- ============================================================

WITH precio_por_zona AS (
    SELECT 
        zona,
        COUNT(*) as cantidad,
        ROUND(AVG(precio), 2) as precio_promedio
    FROM bronze_EDA
    WHERE precio > 0 
      AND zona IS NOT NULL
      AND tipo_de_operacion = 'alquiler'
      AND moneda = 'ARS'
    GROUP BY zona
    HAVING COUNT(*) >= 50
),
zonas_rankeadas AS (
    SELECT 
        zona,
        precio_promedio,
        cantidad as propiedades,
        ROW_NUMBER() OVER (ORDER BY precio_promedio DESC) as ranking
    FROM precio_por_zona
)
SELECT 
    ranking,
    zona,
    precio_promedio,
    propiedades
FROM zonas_rankeadas
WHERE ranking <= 10
ORDER BY ranking;


ranking,zona,moneda,propiedades,precio_promedio,precio_min,precio_max
1,san-isidro,ARS,13289,1049414.7,220000.0,2800000.0
2,gba-zona-norte--san-isidro,ARS,890,997175.28,250000.0,2800000.0
3,vicente-lopez,ARS,21666,995630.18,170000.0,2800000.0
4,bsas-gba-norte/vicente-lopez,ARS,1378,891761.25,220000.0,2800000.0
5,bsas-gba-norte/san-isidro,ARS,987,891252.28,270000.0,2800000.0
6,tigre,ARS,10037,834883.53,165000.0,2800000.0
7,capital-federal,ARS,192245,794349.47,160000.0,2800000.0
8,san-fernando,ARS,7656,789178.42,200000.0,2800000.0
9,gba-zona-norte--tigre,ARS,423,780501.18,250000.0,2500000.0
10,escobar,ARS,4951,773381.54,170000.0,2625000.0


In [0]:
%sql
-- ============================================================
-- EDA 6.2: Comparar precio por zona con el promedio general
-- Usar AVG() OVER() para calcular promedio global sin agrupar
-- ============================================================

WITH precio_por_zona AS (
    SELECT 
        zona,
        COUNT(*) as cantidad,
        ROUND(AVG(precio), 2) as precio_promedio_zona
    FROM bronze_EDA
    WHERE precio > 0 
      AND zona IS NOT NULL
      AND tipo_de_operacion = 'alquiler'
      AND moneda = 'ARS'
    GROUP BY zona
    HAVING COUNT(*) >= 30
)
SELECT 
    zona,
    cantidad as propiedades,
    precio_promedio_zona,
    ROUND(AVG(precio_promedio_zona) OVER (), 2) as precio_promedio_general,
    ROUND(precio_promedio_zona - AVG(precio_promedio_zona) OVER (), 2) as diferencia,
    ROUND((precio_promedio_zona - AVG(precio_promedio_zona) OVER ()) * 100.0 / AVG(precio_promedio_zona) OVER (), 2) as porcentaje_diferencia
FROM precio_por_zona
ORDER BY precio_promedio_zona DESC
LIMIT 15;


ranking,zona,propiedades,m2_promedio,m2_mediana
1,vicente-lopez,21219,15450.67,15000.0
2,bsas-gba-norte/vicente-lopez,1337,15101.13,14687.5
3,bsas-gba-norte/san-isidro,972,14819.01,14545.45
4,capital-federal,188419,14410.29,14130.43
5,san-isidro,13282,14292.01,14393.94
6,gba-zona-norte--san-isidro,890,13900.3,13967.22
7,san-fernando,7374,13264.17,13333.33
8,tigre,9772,13174.09,13066.67
9,bsas-gba-norte/san-fernando,556,12991.38,12931.03
10,bsas-gba-oeste/castelar,44,12610.3,12558.48


In [0]:
%sql
-- ============================================================
-- EDA 6.3: Análisis temporal — agrupar por mes
-- ============================================================

SELECT 
    DATE_TRUNC('month', fecha) as mes,
    COUNT(*) as cantidad_propiedades,
    ROUND(AVG(precio), 2) as precio_promedio
FROM bronze_EDA
WHERE fecha IS NOT NULL
  AND precio > 0
  AND moneda = 'ARS'
  AND tipo_de_operacion = 'alquiler'
GROUP BY DATE_TRUNC('month', fecha)
ORDER BY mes;


moneda,segmento,cantidad,percent_del_mercado,amb_prom,m2_prom,precio_prom
ARS,Económico,6025,1.64%,1.4,81.8,252402.0
ARS,Medio,80111,21.82%,1.5,47.6,410480.0
ARS,Alto,156909,42.73%,2.0,82.0,622774.0
ARS,Premium,124156,33.81%,3.1,266.2,1186424.0
USD,Económico,4310,1.77%,1.5,52.3,436.0
USD,Medio,52714,21.60%,2.2,168.5,719.0
USD,Alto,82703,33.89%,3.5,1362.1,1398.0
USD,Premium,104311,42.74%,4.7,6633.1,10759.0


In [0]:
%sql
-- ============================================================
-- Extra: Análisis de tendencia con LAG (Window Function)
-- Evolución de precios promedio por mes vs mes anterior
-- ============================================================

WITH precios_mensuales AS (
    SELECT 
        DATE_TRUNC('month', fecha) as mes,
        moneda,
        COUNT(*) as cantidad,
        ROUND(AVG(precio), 2) as precio_promedio
    FROM bronze_EDA
    WHERE precio > 0
      AND tipo_de_operacion = 'alquiler'
      AND fecha IS NOT NULL
    GROUP BY DATE_TRUNC('month', fecha), moneda
),
con_variacion AS (
    SELECT 
        mes,
        moneda,
        cantidad,
        precio_promedio,
        -- LAG: obtener valor del mes anterior
        LAG(precio_promedio) OVER (PARTITION BY moneda ORDER BY mes) as precio_mes_anterior,
        -- Calcular variación porcentual
        ROUND(
            (precio_promedio - LAG(precio_promedio) OVER (PARTITION BY moneda ORDER BY mes)) 
            * 100.0 / LAG(precio_promedio) OVER (PARTITION BY moneda ORDER BY mes), 
            2
        ) as variacion_pct
    FROM precios_mensuales
)
SELECT 
    mes,
    moneda,
    cantidad as propiedades,
    precio_promedio,
    precio_mes_anterior,
    COALESCE(variacion_pct || '%', 'N/A') as var_vs_mes_ant
FROM con_variacion
WHERE moneda IN ('ARS', 'USD')
ORDER BY moneda, mes;


mes,moneda,propiedades,precio_promedio,precio_mes_anterior,var_vs_mes_ant
2025-07-01T00:00:00.000Z,ARS,57240,620986.18,null,N/A
2025-08-01T00:00:00.000Z,ARS,39373,657658.33,620986.18,5.91%
2025-09-01T00:00:00.000Z,ARS,29427,666696.87,657658.33,1.37%
2025-10-01T00:00:00.000Z,ARS,31224,677317.43,666696.87,1.59%
2025-11-01T00:00:00.000Z,ARS,149612,843363.45,677317.43,24.52%
2025-12-01T00:00:00.000Z,ARS,55624,852239.76,843363.45,1.05%
2026-01-01T00:00:00.000Z,ARS,4701,773486.7,852239.76,-9.24%
2025-07-01T00:00:00.000Z,USD,19483,8228.33,null,N/A
2025-08-01T00:00:00.000Z,USD,13789,8308.62,8228.33,0.98%
2025-09-01T00:00:00.000Z,USD,10528,8107.56,8308.62,-2.42%


In [0]:
%sql
-- ============================================================
-- Extra: Dashboard Ejecutivo con múltiples CTEs (BONUS)
-- Resumen completo del dataset en una sola query
-- Nota: Material adicional, no está en el PDF de ejercicios
-- ============================================================

WITH metricas_generales AS (
    SELECT 
        COUNT(*) as total_propiedades,
        COUNT(DISTINCT zona) as zonas_unicas,
        MIN(fecha) as fecha_min,
        MAX(fecha) as fecha_max
    FROM bronze_EDA
),
dist_operacion AS (
    SELECT 
        tipo_de_operacion,
        COUNT(*) as cantidad
    FROM bronze_EDA
    GROUP BY tipo_de_operacion
),
dist_moneda AS (
    SELECT 
        moneda,
        COUNT(*) as cantidad,
        ROUND(AVG(precio), 2) as precio_promedio
    FROM bronze_EDA
    WHERE precio > 0
    GROUP BY moneda
),
calidad AS (
    SELECT 
        ROUND(SUM(CASE WHEN precio IS NULL OR precio <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_precio_invalido,
        ROUND(SUM(CASE WHEN metros_cuadrados_totales IS NULL OR metros_cuadrados_totales <= 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_m2_invalido,
        ROUND(SUM(CASE WHEN antiguedad = '999' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) as pct_antiguedad_999
    FROM bronze_EDA
)
SELECT 
    '📊 RESUMEN DEL DATASET' as seccion,
    mg.total_propiedades,
    mg.zonas_unicas,
    mg.fecha_min,
    mg.fecha_max,
    c.pct_precio_invalido || '%' as precio_invalido,
    c.pct_m2_invalido || '%' as m2_invalido,
    c.pct_antiguedad_999 || '%' as antiguedad_999
FROM metricas_generales mg
CROSS JOIN calidad c;


seccion,total_propiedades,zonas_unicas,fecha_min,fecha_max,precio_invalido,m2_invalido,antiguedad_999
📊 RESUMEN DEL DATASET,1197911,99,2025-07-08,2026-01-09,0.01%,4.02%,99.36%


# ================================================================
# PARTE 7: CONCLUSIONES Y PRÓXIMOS PASOS
# ================================================================
# >>> Corresponde a: Ejercicio 7.1 del PDF (Resumen ejecutivo)

## 📋 Hallazgos del EDA

**Resumen ejecutivo:**
- **Total de registros analizados:** 509,395 propiedades
- **Período:** julio 2025 - enero 2026
- **Zonas únicas:** 98
- **Datos válidos (precio > 0):** ~99.6% — la mayoría de registros tienen precio
- **Datos con problemas:** expensas (77.8% nulos), orientación (96.1% nulos), antigüedad (98.9% con valor 999)

### Problemas de calidad identificados:

1. **Valores placeholder:** El valor `999` en antigüedad representa datos faltantes
2. **Campos nulos:** Varios campos como `expensas`, `orientacion`, `cochera` tienen alto % de nulos
3. **Datos mixtos:** Precios en USD y ARS mezclados (requiere normalización)
4. **Posibles duplicados:** Propiedades repetidas con misma ubicación y precio
5. **Outliers:** Precios extremadamente altos o bajos que podrían ser errores

### Transformaciones necesarias para capa Silver:

- [ ] Convertir `999` en antigüedad a NULL
- [ ] Normalizar precios a una sola moneda (o crear columnas separadas)
- [ ] Eliminar duplicados
- [ ] Filtrar outliers extremos
- [ ] Parsear el campo ubicación para extraer barrio/ciudad
- [ ] Calcular métricas derivadas (precio por m2)

---

## 🎯 Próxima semana: Arquitectura Medallion

En la Semana 3 aprenderemos a:
- Crear la capa **Silver** con datos limpios
- Aplicar transformaciones de calidad
- Implementar el patrón Bronze → Silver → Gold

---

## ✅ Checklist de finalización

- [ ] Creaste tu catálogo personal
- [ ] Creaste el esquema `raw` 
- [ ] Creaste el volume y subiste el CSV
- [ ] Creaste la tabla `properties_bronze`
- [ ] Ejecutaste todas las queries de EDA
- [ ] Identificaste al menos 5 problemas de calidad
- [ ] Entendés la diferencia entre tablas MANAGED y EXTERNAL

---

*Bootcamp: Fundamentos de Ingeniería de Datos | Instructor: Luciano Argolo | [lucianoargolo.com](https://lucianoargolo.com)*
